In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, IntegerType, DateType, StringType, DoubleType

In [0]:
exchange_rates_df = spark.table("02_dev_silver.transformed.exchange_rates")
customers_df=spark.table("02_dev_silver.transformed.customers")
order_items_df = spark.table("02_dev_silver.transformed.order_items")
orders_df = spark.table("02_dev_silver.transformed.orders")
products_df = spark.table("02_dev_silver.transformed.products")

In [0]:
dim_customer = customers_df.withColumn(
    "customer_key", row_number().over(Window.orderBy("customer_id"))
).select(
    "customer_key",
    "customer_id",
    "customer_name",
    "email_address",
    "country",
    "channel",
    "registration_date"
)

dim_customer = dim_customer.join(
    dim_date.select(col("full_date").alias("registration_date"), "date_key"),
    "registration_date",
    "left"
).withColumn(
    "registration_date_key",
    when(col("date_key").isNull(), -1).otherwise(col("date_key"))
).select(
    "customer_key",
    "customer_id",
    "customer_name",
    "email_address",
    "country",
    "channel",
    col("registration_date_key")
)

unknown_customer_schema = StructType([
    StructField("customer_key", IntegerType(), False),
    StructField("customer_id", StringType(), True),
    StructField("customer_name", StringType(), True),
    StructField("email_address", StringType(), True),
    StructField("country", StringType(), True),
    StructField("channel", StringType(), True),
    StructField("registration_date_key", IntegerType(),True),
])

unknown_customer = spark.createDataFrame([
    Row(
        customer_key=-1,
        customer_id="Unknown",
        customer_name="Unknown",
        email_address="Unknown",
        country="Unknown",
        channel="Unknown",
        registration_date_key=-1
    )
], schema=unknown_customer_schema)

dim_customer = dim_customer.union(unknown_customer)

In [0]:
dim_product = products_df.withColumn(
    "product_key", row_number().over(Window.orderBy("product_id"))
).select(
    "product_key",
    "product_id",
    "product_name",
    "category",
    "price",
    "currency",
    "country_code",
    "exchange_rate_to_usd",
    "base_price"
)

unknown_product_schema = StructType([
    StructField("product_key", IntegerType(), False),
    StructField("product_id", StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("currency", StringType(), True),
    StructField("country_code", StringType(), True),
    StructField("exchange_rate_to_usd", DoubleType(), True),
    StructField("base_price", DoubleType(), True)
])

unknown_product = spark.createDataFrame([
    Row(
        product_key=-1,
        product_id="Unknown",
        product_name="Unknown",
        category="Unknown",
        price=None,
        currency="Unknown",
        country_code="Unknown",
        exchange_rate_to_usd=None,
        base_price=None
    )
], schema=unknown_product_schema)

dim_product = dim_product.union(unknown_product)

In [0]:
order_dates = orders_df.select(col("order_date").alias("date"))
customer_dates = customers_df.select(col("registration_date").alias("date"))

dim_date = order_dates.union(customer_dates).dropna().dropDuplicates()

dim_date = dim_date.withColumn(
    "date_key",
    date_format(col("date"), "yyyyMMdd").cast("int")
)

dim_date = dim_date.select(
    "date_key",
    col("date").alias("full_date"),
)

unknown_date_schema = StructType([
    StructField("date_key", IntegerType(), False),
    StructField("full_date", DateType(), True)
])

unknown_date = spark.createDataFrame([
    Row(date_key=-1, full_date=None)
], schema=unknown_date_schema)

dim_date = dim_date.union(unknown_date)

In [0]:
fact_df = order_items_df.join(
    orders_df.select("order_id", "customer_id", "order_date", "order_status"),
    "order_id",
    "left"
)

fact_df = fact_df.join(
    dim_customer.select("customer_id", "customer_key"),
    "customer_id",
    "left"
)

fact_df = fact_df.join(
    dim_product.select("product_id", "product_key"),
    "product_id",
    "left"
)

fact_df = fact_df.withColumn(
    "date_key",
    when(col("order_date").isNull(), -1)
    .otherwise(date_format(col("order_date"), "yyyyMMdd").cast("int"))
).withColumn(
    "customer_key",
    when(col("customer_key").isNull(), -1).otherwise(col("customer_key"))
).withColumn(
    "product_key",
    when(col("product_key").isNull(), -1).otherwise(col("product_key"))
)

fact_order_items = fact_df.select(
    "order_item_id",
    "order_id",
    "customer_key",
    "product_key",
    "date_key",
    "quantity",
    "base_unit_price",
    "base_line_total",
    "order_status"
)

In [0]:
table_names = {
    "02_dev_silver.facts_and_dims.dim_customer": dim_customer,
    "02_dev_silver.facts_and_dims.dim_product": dim_product,
    "02_dev_silver.facts_and_dims.dim_date": dim_date,
    "02_dev_silver.facts_and_dims.fact_order_items": fact_order_items
}

for table_name, df in table_names.items():
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_name)